# Environment Setup & Model Loading

This notebook covers the environment and model-loading stage and ends with a smoke test that confirms the inference path works.

**Colab GPU:** Prefer L4 or A100

**Colab runtime version:** pin to `2026.07` under Runtime > Change runtime type.


## 1. Check the Runtime and GPU

The interpreter version is checked first because it decides whether the pinned
dependencies below can be installed at all. `tokenizers==0.19.1`, which OpenVLA's
remote modelling code requires, ships no wheel beyond CPython 3.12 and cannot be
built from source in the runtime, so on a Python 3.13 runtime the install fails and
the session silently keeps the pre-installed transformers, whose 5.x releases no
longer expose `AutoModelForVision2Seq`. Pinning the Colab runtime version keeps the
interpreter at 3.12 and makes the mismatch impossible rather than merely detectable.


In [1]:
import sys
import torch

REQUIRED_PYTHON = (3, 12)          # highest version with wheels for the pinned set
COLAB_RUNTIME_VERSION = '2026.07'  # last runtime version shipping that interpreter

assert sys.version_info[:2] == REQUIRED_PYTHON, (
    f'Python {sys.version_info.major}.{sys.version_info.minor} is active, but the pinned '
    f'dependencies require Python {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}. Set Runtime > '
    f'Change runtime type > Runtime version to {COLAB_RUNTIME_VERSION}, then reconnect and '
    f'run this notebook from the top.'
)
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU.'
p = torch.cuda.get_device_properties(0)
print(f'Python {sys.version.split()[0]}  torch {torch.__version__}')
print(f'GPU: {p.name}  sm_{p.major}{p.minor}  {p.total_memory/1024**3:.1f} GB')
print('FlashAttention-2 / native bf16 available:' , (p.major, p.minor) >= (8, 0))

Python 3.12.13  torch 2.10.0+cu128
GPU: NVIDIA A100-SXM4-40GB  sm_80  39.5 GB
FlashAttention-2 / native bf16 available: True


## 2. Mount Google Drive

Caching the HF weights in Drive.


In [2]:
from google.colab import drive, files
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF cache ->', os.environ['HF_HOME'])

Mounted at /content/drive
HF cache -> /content/drive/MyDrive/openvla_cache/hf


## 3. Install Pinned Dependencies

> Do **not** reinstall torch, Colab's build is matched to its CUDA driver.

`--only-binary=:all:` forbids a source build, so a missing wheel stops the install
instead of failing late inside a compiler and leaving the pre-installed versions in
place.


In [3]:
!pip install -q --only-binary=:all: transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 \
    huggingface_hub==0.23.4 accelerate==0.30.1 'bitsandbytes>=0.45.0' \
    --upgrade "protobuf>=6.31.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 96.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 135.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 73.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.4/340.4 kB 41.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.23.4 whi

**After this install, restart the runtime once** because Colab pre-imports a newer transformers. Then re-run the runtime check, skip this install cell, and run the verification below.


In [4]:
from importlib.metadata import version

# OpenVLA's remote modelling code raises on an unsupported timm and only warns about a
# transformers or tokenizers mismatch, which would otherwise be lost in the load output.
EXPECTED = {
    'transformers': '4.40.1',
    'tokenizers': '0.19.1',
    'timm': '0.9.10',
    'huggingface_hub': '0.23.4',
    'accelerate': '0.30.1',
}
installed = {name: version(name) for name in EXPECTED}
for name, found in installed.items():
    print(f'{name:16s} {found}')

mismatched = {n: v for n, v in installed.items() if v != EXPECTED[n]}
assert not mismatched, (
    f'Installed versions differ from the pin: '
    + ', '.join(f'{n} {v} (expected {EXPECTED[n]})' for n, v in mismatched.items())
    + '. Re-run the install cell, check it reported no error, and restart the runtime.'
)

transformers     4.40.1
tokenizers       0.19.1
timm             0.9.10
huggingface_hub  0.23.4
accelerate       0.30.1


## 4. Import from `model.py`

Clones the project code from GitHub into the runtime and imports the loader,
inference, and logging functions from there, so the code always matches the
pushed commit.

In [5]:
import sys, os, glob, importlib, subprocess

REPO_URL = 'https://github.com/LewisTL/ECS8056.git'
BRANCH = 'master'
REPO_DIR = '/content/ECS8056'

def sync_repo():
    """Clone or hard-refresh the repository so it matches origin/BRANCH."""
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url],
                       check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--depth', '1',
                        'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '--quiet',
                        f'origin/{BRANCH}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch',
                        BRANCH, url, REPO_DIR], check=True)
    return subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()


commit = sync_repo()
module_dir = REPO_DIR
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for _m in ('model', 'data', 'export_pairs', 'detect_duplicates'):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()

from model import load_openvla, predict_action, run_metadata, append_prediction_log
print(f'imported model.py from {module_dir} @ {commit}')

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

imported model.py from /content/ECS8056 @ a9f94b2


## 5. Load OpenVLA-7B

In [6]:
processor, vla, compute_dtype = load_openvla(quantize_4bit=True, precision='bf16')
meta = run_metadata(compute_dtype)   # GPU, dtype, seed, library versions for logging
print(meta)

[load_openvla] GPU: NVIDIA A100-SXM4-40GB (sm_80, 39.5 GB) | precision=bf16 | attn=eager | 4bit=True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:99: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[load_openvla] Loaded. GPU memory allocated: 4.08 GB
{'gpu_name': 'NVIDIA A100-SXM4-40GB', 'gpu_capability': 'sm_80', 'dtype': 'bfloat16', 'seed': 42, 'torch': '2.10.0+cu128', 'transformers': '4.40.1', 'bitsandbytes': '0.50.1'}


## 6. Smoke Test

A synthetic frame is sufficient here; the only check is that the pipeline produces
a well-formed 7-DoF action `[dx, dy, dz, droll, dpitch, dyaw, gripper]`. Real
BridgeData V2 frames come in the next notebook.


In [7]:
import numpy as np
from PIL import Image

CSV_PATH = '/content/drive/MyDrive/openvla_cache/predictions.csv'
dummy = Image.fromarray((np.random.default_rng(0).random((224,224,3))*255).astype(np.uint8))

instr = 'pick up the object on the left'
action = predict_action(processor, vla, dummy, instr, compute_dtype)
append_prediction_log(CSV_PATH, action, instr, meta, scene_id='smoke', spatial_term='left')

print('Action shape :', action.shape)
print('Action vector:', np.round(action, 4))
assert action.shape == (7,), 'Expected a 7-DoF vector'


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Action shape : (7,)
Action vector: [-0.0029  0.0161 -0.006  -0.0014  0.0086  0.203   0.    ]


## 6. Sanity Check

Same image, two instructions differing only in the spatial term. With a random
image, a clean sign flip should not be expected. This cell just confirms the
probe mechanics (two calls, compare `dx`) work before real scenes are wired in.


In [8]:
left  = predict_action(processor, vla, dummy, 'move to the cup on the left',  compute_dtype)
right = predict_action(processor, vla, dummy, 'move to the cup on the right', compute_dtype)
print('dx(left) = %.4f   dx(right) = %.4f   diff = %.4f' % (left[0], right[0], left[0]-right[0]))

dx(left) = -0.0078   dx(right) = 0.0031   diff = -0.0110
